# Notebook 05a — Global-Bus LightGBM (Direct Bus-Level Forecasting)

## Purpose

Implement the second of two ML forecasting strategies the assignment hints at: **bus-level direct forecasting with a single pooled LightGBM model**. In contrast to notebook 04's zone-direct + top-down approach (16 zone-specific models + hour-of-day disaggregation), this notebook trains exactly two models — one per task — that predict each bus's `pd` directly from bus-level features.

The pipeline is one stage rather than two:

1. **Direct bus-level forecasting.** Train one LightGBM model per task (2 total) on the full pooled bus-level feature matrix (~95M training rows × 4,208 buses). `bus_unique_id` and `zone_name` are passed as categorical features. The model learns bus-specific and zone-specific patterns jointly with the lag, calendar, and event features, and predicts bus-level pd at hour t directly.

This strategy is anchored on Pinheiro et al. 2023, who applied global LightGBM to 96,989 Portuguese distribution substations and reported competitive accuracy against zone-aggregation approaches. It also serves as the methodological complement to Triebe et al. 2025 (MISO), who tested both approaches and found zone-direct + top-down preferable on long-horizon tasks but global-bus competitive on short-horizon.

This notebook addresses **Q1** of the project's research questions: *zone-direct + top-down vs global-bus direct, which approach wins on which task?* Notebook 04 produced the zone-direct submission; this notebook produces the global-bus comparator; notebook 06 evaluates them side-by-side.

## Scope of this notebook

This notebook does NOT:
- Train zone-specific models (that was notebook 04)
- Use deep learning (notebooks 05b PatchTST and 05c NHITS)
- Compute evaluation metrics (notebook 06)

This notebook DOES:
- Load the full bus-level feature matrices for both tasks from notebook 02
- Train two pooled LightGBM models (one per task) on ~95M rows each with `bus_unique_id` and `zone_name` as categorical features
- Run Optuna hyperparameter search (10 trials per task, 20 total)
- Generate bus-level predictions for all 2025 test rows
- Write two forecast parquet files in the assignment's required schema

## Methodological decisions (locked in)

| Decision | Choice | Rationale |
|---|---|---|
| **Bus identity** | `bus_unique_id` as LightGBM categorical | Captures bus-specific bias terms that lag features alone don't encode. Triebe et al. 2025 used this approach; LightGBM handles 4,208-category cardinality via its native categorical-split algorithm. |
| **Zone identity** | `zone_name` as LightGBM categorical | Captures zone-level structural patterns (industrial FWES, urban NCEN, growth-zone behavior) that the per-row zone-level lag features partially but not fully encode. |
| **Feature set** | Full set from notebook 02 (29 model features per task) | No ablation; LightGBM's `feature_fraction` provides effective subsampling during training. |
| **Optuna budget** | 10 trials per task (20 total) | Tradeoff between thoroughness and compute. At 95M training rows, each trial costs 3-10 minutes; 10 trials per task captures most of the optimization gain without spending 6+ hours on diminishing returns. |
| **Train/validation/test split** | Identical to notebook 04 | Train 2022-2023, validate 2024 for Optuna, final-train on 2022-2024, test 2025. Direct reproducibility with notebook 04 enables clean Q1 comparison. |
| **Memory strategy** | Pandas load → LightGBM Dataset with `free_raw_data=True` | Releases the pandas DataFrame memory after the LightGBM Dataset is constructed. Peak memory ~12-15 GB, fits comfortably in 16 GB systems. |
| **Cold-start handling** | Implicit via LightGBM's unseen-category routing | The 42 cold-start buses appear in the test set with `bus_unique_id` values the model never saw in training. LightGBM routes unknown categories to a default direction at each split, effectively using zone-level + lag-feature signal. This matches notebook 04's share-fallback in spirit. |

## Feature engineering

We use the feature matrices from notebook 02 directly — no recomputation needed. The 32-column matrices (verified via diagnostic before this cell) contain:

**Bus-level features:**
- Identity: `bus_unique_id` (4,208 categories), `zone_name` (8 categories), `timestamp` (split marker, not a feature)
- Target: `pd` (bus-level load in MW)
- Calendar coordinates: `year`, `month`, `day`, `dow`, `hour`
- Cyclical encoding: `hour_sin`, `hour_cos`, `dow_sin`, `dow_cos`, `month_sin`, `month_cos`
- Event flags: `is_weekend`, `is_holiday`, `is_winter_storm_elliott`
- Bus-level autoregressive lags (next-day): `pd_lag_{24h, 48h, 168h, 336h, 720h, 8760h}`
- Bus-level trailing means (next-day): `pd_trailing_mean_{24h, 168h}_at_fc`

**Zone-level features (pre-computed by notebook 02, available at each bus row):**
- `zone_load_bus_count`, `zone_gen_bus_count` (zone activity counts, vary over time)
- `zone_pd_lag_24h` (zone aggregate 24h ago — gives model zone-level recent context)
- `zone_pd_trailing_mean_{24h, 168h}_at_fc` (zone aggregate trailing means)

**Split marker (not a feature):**
- `is_test_period` (boolean, True for 2025 rows)

Total: 32 columns, of which 29 are model features (excluding `timestamp`, `pd`, `is_test_period`).

The next-month feature matrix has the same structure with task-appropriate lags (1440h, 2160h, 8760h, 17520h) and trailing means (30d, 90d). We will verify the exact next-month schema in Cell 2's diagnostic.

All lag and trailing-mean features respect the forecast_created_at constraint — same admissibility logic as notebooks 02 and 04.

**Target column:** bus-level `pd` in raw MW. No log transform (same choice as notebook 04).

## How this compares to notebook 04

| Property | Notebook 04 (zone-direct) | Notebook 05a (global-bus) |
|---|---|---|
| Number of models | 16 (8 zones × 2 tasks) | 2 (1 per task) |
| Training rows per model | ~17K-26K | ~95M |
| Target | Zone-aggregated pd | Bus-level pd |
| Bus identity in model | Implicit (shares applied post-hoc) | Explicit (categorical feature) |
| Disaggregation step | Yes (hour-of-day shares) | No (direct bus predictions) |
| Cold-start treatment | Share-fallback (explicit) | Categorical-routing (implicit) |
| Zone-level features | Built from zone series | Pre-joined at bus row by notebook 02 |
| Total Optuna trials | 240 (15 × 16) | 20 (10 × 2) |
| Approx compute | 5 min | 2-4 hours |

The same train/val/test split, target column, and feature philosophy apply in both notebooks. The substantive difference is how the model sees the data: notebook 04 sees one zone at a time and infers the bus distribution post-hoc; notebook 05a sees all buses and zones at once and learns the joint structure directly.

## Outputs

Two forecast parquet files written to `data/processed/forecasts/`, matching the assignment schema:

| File | Task | model_name |
|---|---|---|
| `forecast_global_bus_lgbm_nextday.parquet` | Next-day | `global_bus_lgbm_nextday` |
| `forecast_global_bus_lgbm_nextmonth.parquet` | Next-month | `global_bus_lgbm_nextmonth` |

Each file: 32,427,554 rows in the 7-column required schema, matching notebook 03 and notebook 04 row-for-row for clean comparison in notebook 06.

We also write best-hyperparameter JSON files for documentation:
- `data/processed/model_params/best_params_global_bus_nextday.json`
- `data/processed/model_params/best_params_global_bus_nextmonth.json`

This is a new directory; we create it in Cell 2. The naming generalizes to accommodate notebooks 05b and 05c if those get implemented.

## Cold-start handling — methodological note

The 42 cold-start buses present in the 2025 test set never appeared in training (2022-2024). LightGBM's categorical-feature implementation handles this case via a documented mechanism: at each tree split that uses `bus_unique_id`, the algorithm chooses how to route categories not seen during training. The default (since LightGBM 2.x) is to route them to the side that minimizes loss on the training data, effectively giving cold-start buses predictions based on the zone-level and feature-level signal that survives once the bus-specific bias is unavailable.

This is methodologically equivalent in spirit to notebook 04's zone-hour share-fallback: when bus-specific signal is unavailable, fall back to zone-level patterns. We expect cold-start bus predictions to be somewhat less accurate than well-trained-bus predictions, but defensible. Notebook 06 will report cold-start vs non-cold-start performance separately.

## Runtime estimate

Approximately 2-4 hours total, dominated by Optuna search.

| Stage | Time |
|---|---|
| Load 8 feature parquet files | 2-5 min |
| Optuna search (20 trials at 3-10 min each) | 1-3 hours |
| Final retraining (2 models on 2022-2024) | 10-30 min |
| Generate 2025 predictions + write files | 10-15 min |
| Verification | 2 min |

This is the most compute-intensive notebook in the pipeline. Plan to run in a single session with the laptop plugged in. The runtime depends heavily on per-trial early-stopping behavior — if the models converge in 200-300 trees, the lower bound applies; if they need 1000+ trees, expect the upper bound.

Unlike notebook 04, the Optuna search here is NOT cheap to re-run. The checkpoint mechanism is essential — if the kernel dies mid-search, re-running the notebook resumes from where it left off via the JSON cache.

In [1]:
"""
Imports, configuration, and path setup for notebook 05a.

This cell establishes the runtime environment for the global-bus LightGBM pipeline:
  - Standard library: pathlib for portable paths, warnings to suppress benign
    LightGBM/Optuna info messages, gc and time for memory management and runtime
    instrumentation, json for writing best-params artifacts, psutil for monitoring
    available RAM.
  - Numeric/data stack: numpy, pandas, pyarrow for column-selective parquet reads.
  - ML stack: lightgbm for the gradient-boosted tree models, optuna for the
    hyperparameter search.

Path conventions: this notebook lives in assignment2/notebooks/. Inputs come from
data/processed/features/ (notebook 02 outputs). Forecast outputs land in
data/processed/forecasts/ alongside notebooks 03 and 04. Best-hyperparameter
artifacts land in a new data/processed/model_params/ subdirectory (created here).

If lightgbm or optuna are not installed, the cell fails with a clear message.
"""

# Standard library
from pathlib import Path
import warnings
import gc
import time
import json
import psutil

# Numeric and data
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

# ML
try:
    import lightgbm as lgb
except ImportError as e:
    raise ImportError(
        "lightgbm is required for notebook 05a. Install with: pip install lightgbm. "
        "On macOS, if the install succeeds but import fails with an OpenMP error, "
        "run: brew install libomp"
    ) from e

try:
    import optuna
    from optuna.samplers import TPESampler
except ImportError as e:
    raise ImportError(
        "optuna is required for notebook 05a. Install with: pip install optuna"
    ) from e

# Display and warning configuration
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
warnings.simplefilter("ignore", category=FutureWarning)

# Quiet Optuna's per-trial logging — we'll print our own concise summaries
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Paths (relative to notebook location: assignment2/notebooks/)
DATA_DIR = Path("../data")
AUDIT_DIR = Path("../data/processed/audit")
FEATURES_DIR = Path("../data/processed/features")
FORECASTS_DIR = Path("../data/processed/forecasts")
MODEL_PARAMS_DIR = Path("../data/processed/model_params")

# Create the new model_params directory; the others already exist
MODEL_PARAMS_DIR.mkdir(parents=True, exist_ok=True)

# Per-year feature file paths from notebook 02
YEARS = [2022, 2023, 2024, 2025]
NEXTDAY_FEATURE_FILES = {y: FEATURES_DIR / f"features_nextday_{y}.parquet" for y in YEARS}
NEXTMONTH_FEATURE_FILES = {y: FEATURES_DIR / f"features_nextmonth_{y}.parquet" for y in YEARS}

# Audit artifact from notebook 01
FORECASTABLE_BUS_LIST_PATH = AUDIT_DIR / "forecastable_bus_list.parquet"

# Verify all expected inputs exist before proceeding
for y in YEARS:
    assert NEXTDAY_FEATURE_FILES[y].exists(), f"Missing: {NEXTDAY_FEATURE_FILES[y]}"
    assert NEXTMONTH_FEATURE_FILES[y].exists(), f"Missing: {NEXTMONTH_FEATURE_FILES[y]}"
assert FORECASTABLE_BUS_LIST_PATH.exists(), (
    f"Missing audit artifact: {FORECASTABLE_BUS_LIST_PATH}. Run notebook 01 first."
)

# Configuration constants for this notebook
TRAIN_YEARS = [2022, 2023]              # Optuna training set
VAL_YEAR = 2024                          # Optuna validation set
FINAL_TRAIN_YEARS = [2022, 2023, 2024]   # Final retrain after Optuna
TEST_YEAR = 2025                         # Forecast target year
N_OPTUNA_TRIALS = 10                     # Per task (20 trials total)
OPTUNA_SEED = 42                         # Reproducibility (matches notebook 04)

# Categorical features for LightGBM
CATEGORICAL_FEATURES = ["bus_unique_id", "zone_name"]

# Columns that exist in the feature parquets but are NOT model features
NON_FEATURE_COLUMNS = ["timestamp", "pd", "is_test_period"]

# Verify the next-month schema matches expectations (next-day was already verified above)
print("Verifying next-month feature schema (2024 sample)...")
nextmonth_sample = pq.read_table(
    NEXTMONTH_FEATURE_FILES[2024],
    columns=None,  # read all columns once for schema check
).schema
nextmonth_cols = [field.name for field in nextmonth_sample]
print(f"\nNext-month feature file columns ({len(nextmonth_cols)} total):")
for c in nextmonth_cols:
    print(f"  {c}")

print(f"\nLightGBM version: {lgb.__version__}")
print(f"Optuna version: {optuna.__version__}")
print(f"\nFeature files located in {FEATURES_DIR.resolve()}")
print(f"Audit artifacts located in {AUDIT_DIR.resolve()}")
print(f"Forecast outputs will be written to {FORECASTS_DIR.resolve()}")
print(f"Hyperparameter artifacts will be written to {MODEL_PARAMS_DIR.resolve()}")
print(f"\nCategorical features: {CATEGORICAL_FEATURES}")
print(f"Train/val/final-train/test years: {TRAIN_YEARS} / {VAL_YEAR} / {FINAL_TRAIN_YEARS} / {TEST_YEAR}")
print(f"Optuna trials per task: {N_OPTUNA_TRIALS}  (total: {N_OPTUNA_TRIALS * 2})")

Verifying next-month feature schema (2024 sample)...

Next-month feature file columns (30 total):
  bus_unique_id
  zone_name
  timestamp
  pd
  year
  month
  day
  dow
  hour
  is_test_period
  hour_sin
  hour_cos
  dow_sin
  dow_cos
  month_sin
  month_cos
  is_weekend
  is_holiday
  is_winter_storm_elliott
  zone_load_bus_count
  zone_gen_bus_count
  pd_lag_1440h
  pd_lag_2160h
  pd_lag_8760h
  pd_lag_17520h
  pd_trailing_mean_30d_at_fc
  pd_trailing_mean_90d_at_fc
  zone_pd_lag_1440h
  zone_pd_trailing_mean_30d_at_fc
  zone_pd_trailing_mean_90d_at_fc

LightGBM version: 4.6.0
Optuna version: 4.8.0

Feature files located in /Users/gavinyu/Desktop/ECESIS Investments Assignments/ECESIS-2026-Summer-Power-Systems-Modeling-Assignment/assignment2/data/processed/features
Audit artifacts located in /Users/gavinyu/Desktop/ECESIS Investments Assignments/ECESIS-2026-Summer-Power-Systems-Modeling-Assignment/assignment2/data/processed/audit
Forecast outputs will be written to /Users/gavinyu/Desk

## Input verification (per-task processing)

Following the methodological refactor, we process the two forecasting tasks **sequentially rather than simultaneously**. The previous approach — loading both task DataFrames into memory before processing — peaked at ~30 GB of pandas-held memory plus 5-8 GB of LightGBM working set, putting the machine into macOS memory compression and grinding the Optuna search to 10× expected runtime.

The refactored pattern is:

1. For each task, load that task's data only (4 parquet files concatenated into a single DataFrame).
2. Build train/val/finaltrain/test splits.
3. Run Optuna search → final-train → predict → write forecast file.
4. Release everything before loading the next task.

Peak memory per task: ~18 GB (one task's DataFrame plus LightGBM Datasets and prediction buffers). Between tasks: ~3-4 GB baseline.

This cell verifies the input files exist and reports their on-disk structure without actually loading them into memory. The actual loading happens inside the per-task loop in Cell 5.

In [2]:
"""
Verify input feature parquets exist and report their structure WITHOUT loading them.

For each of the 8 feature files (4 years × 2 tasks), read just the parquet metadata
to confirm:
  - File exists on disk
  - Schema matches expectations (column count, key columns present)
  - Row count is the canonical value from notebook 02

This replaces the previous Cell 3 (which loaded both task DataFrames into memory). The
new design defers loading until each task is being actively processed, keeping
baseline memory usage at ~3-4 GB.

Runtime: <5 seconds (metadata-only reads).
"""

t0 = time.time()

print("Verifying feature file metadata (no actual data loaded)...\n")

PER_YEAR_ROW_COUNTS = {2022: 32_692_332, 2023: 32_897_778, 2024: 32_962_294, 2025: 32_427_554}
EXPECTED_TOTAL = sum(PER_YEAR_ROW_COUNTS.values())  # 130,979,958

NEXTMONTH_DROP_COLS = ["pd_lag_17520h"]  # consistency with Optuna training window

# Per-task expected column counts (after applying drops)
EXPECTED_COL_COUNTS = {
    "nextday": 32,    # full schema, no drops
    "nextmonth": 29,  # 30 - 1 dropped column
}

# Required identity, target, and split columns
REQUIRED_COLS = ["bus_unique_id", "zone_name", "timestamp", "pd", "is_test_period"]

# Per-task feature file paths (already defined in Cell 2)
TASK_FILES = {
    "nextday": NEXTDAY_FEATURE_FILES,
    "nextmonth": NEXTMONTH_FEATURE_FILES,
}

# ──────────────────────────────────────────────────────────────────────────
# Verify each task's files
# ──────────────────────────────────────────────────────────────────────────
for task, files_dict in TASK_FILES.items():
    print(f"  {task}:")
    task_total = 0
    for y in YEARS:
        path = files_dict[y]
        # Read schema only (no data)
        metadata = pq.read_metadata(path)
        size_mb = path.stat().st_size / 1024**2

        # Pull column names from the schema
        all_cols = [field.name for field in pq.read_schema(path)]
        # Apply task-specific drops to compute the post-drop column count
        if task == "nextmonth":
            kept_cols = [c for c in all_cols if c not in NEXTMONTH_DROP_COLS]
        else:
            kept_cols = all_cols
        n_rows = metadata.num_rows

        # Hard checks
        missing_required = [c for c in REQUIRED_COLS if c not in all_cols]
        assert not missing_required, f"{path.name}: missing required cols {missing_required}"
        assert n_rows == PER_YEAR_ROW_COUNTS[y], (
            f"{path.name}: {n_rows:,} rows, expected {PER_YEAR_ROW_COUNTS[y]:,}"
        )
        assert len(kept_cols) == EXPECTED_COL_COUNTS[task], (
            f"{path.name}: {len(kept_cols)} post-drop cols, expected {EXPECTED_COL_COUNTS[task]}"
        )

        task_total += n_rows
        print(f"    {y}: {n_rows:>11,} rows × {len(kept_cols):>2} cols  ({size_mb:>5.0f} MB on disk)")

    assert task_total == EXPECTED_TOTAL, (
        f"{task}: total {task_total:,}, expected {EXPECTED_TOTAL:,}"
    )
    print(f"    Total:    {task_total:>11,} rows  ✓")

# ──────────────────────────────────────────────────────────────────────────
# Memory baseline (we haven't loaded any data)
# ──────────────────────────────────────────────────────────────────────────
elapsed = time.time() - t0
print(f"\n✓ All 8 feature files verified in {elapsed:.1f}s (metadata-only, no data loaded)")
print(f"  Expected per-task row count: {EXPECTED_TOTAL:,}")
print(f"  Categorical features: {CATEGORICAL_FEATURES}")

mem = psutil.virtual_memory()
print(f"\nSystem RAM available (no feature data loaded): {mem.available / 1024**3:.1f} GB / {mem.total / 1024**3:.1f} GB total")

Verifying feature file metadata (no actual data loaded)...

  nextday:
    2022:  32,692,332 rows × 32 cols  (  558 MB on disk)
    2023:  32,897,778 rows × 32 cols  (  643 MB on disk)
    2024:  32,962,294 rows × 32 cols  (  648 MB on disk)
    2025:  32,427,554 rows × 32 cols  (  638 MB on disk)
    Total:    130,979,958 rows  ✓
  nextmonth:
    2022:  32,692,332 rows × 29 cols  (  324 MB on disk)
    2023:  32,897,778 rows × 29 cols  (  440 MB on disk)
    2024:  32,962,294 rows × 29 cols  (  508 MB on disk)
    2025:  32,427,554 rows × 29 cols  (  503 MB on disk)
    Total:    130,979,958 rows  ✓

✓ All 8 feature files verified in 0.0s (metadata-only, no data loaded)
  Expected per-task row count: 130,979,958
  Categorical features: ['bus_unique_id', 'zone_name']

System RAM available (no feature data loaded): 25.1 GB / 36.0 GB total


### Input verification — observations

All 8 feature files exist on disk with the expected structure:

| Task | Year | Rows | Cols (post-drop) | Disk size |
|---|---|---|---|---|
| nextday | 2022 | 32,692,332 | 32 | 558 MB |
| nextday | 2023 | 32,897,778 | 32 | 643 MB |
| nextday | 2024 | 32,962,294 | 32 | 648 MB |
| nextday | 2025 | 32,427,554 | 32 | 638 MB |
| nextmonth | 2022 | 32,692,332 | 29 | 324 MB |
| nextmonth | 2023 | 32,897,778 | 29 | 440 MB |
| nextmonth | 2024 | 32,962,294 | 29 | 508 MB |
| nextmonth | 2025 | 32,427,554 | 29 | 503 MB |

Each task totals 130,979,958 rows — identical to notebook 02's canonical row count and consistent across notebooks 03 and 04. The per-task column counts (32 nextday, 29 nextmonth) reflect the schema differences in lag horizons plus the `pd_lag_17520h` drop applied to nextmonth.

The on-disk size asymmetry (nextday 2.49 GB vs nextmonth 1.78 GB) is driven by both the smaller column count and the lower per-column entropy of the longer-horizon features — month-ago lag values change less hour-to-hour than 24-hour lags, compressing more effectively under parquet's default zstd encoding.

**Baseline memory: 23.9 GB available out of 36 GB total.** This is the clean starting state for the per-task loop. With this headroom, loading one task's data (~13 GB) plus constructing LightGBM Datasets (~5 GB) plus working memory (~2-3 GB) gives a peak of ~21 GB used, leaving 15+ GB of physical RAM free for the OS and other processes. No memory compression, no swap, no thrashing.

The 36 GB total RAM is a meaningful asset for this assignment. The previous attempt loaded both tasks (~24 GB), pushed peak usage past 30 GB, and triggered macOS memory compression — that's what caused the 10-20× training slowdown we observed in trial 2 of the interrupted Optuna run. The refactor's per-task pattern keeps peak usage well within physical RAM throughout.

## Train/validation/test split

We apply the same expanding-window holdout split as notebook 04, for the same methodological reasons (Hyndman & Athanasopoulos, Ch. 5; Bergmeir & Benítez 2012; Triebe et al. 2025):

| Slice | Years | Use |
|---|---|---|
| **Train** | 2022-2023 | Optuna search training fold (~65M rows) |
| **Validation** | 2024 | Optuna trial scoring (~33M rows) |
| **Final-train** | 2022-2024 | Refit with best params before generating 2025 predictions (~98M rows) |
| **Test** | 2025 | Generate predictions to write to forecast files (~32M rows) |

**Row counts are much larger than notebook 04.** Where the zone-direct model trained on ~17K rows per zone-task, the global-bus model trains on ~65M-98M rows per task. The signal-to-noise ratio and training dynamics are different at this scale — convergence happens with more boosting iterations, individual feature contributions are smaller, and overfitting risk shifts.

**NaN handling unchanged from notebook 04.** Lag features with insufficient lookback (e.g., `pd_lag_8760h` for 2022 rows) remain as NaN. LightGBM's native missing-value handling routes them via learned split direction (Ke et al., 2017). No row-dropping or imputation.

**2025-12-04 systematically missing.** The test slice has 32,427,554 rows, same as notebooks 03 and 04. This carries through naturally.

**Memory management.** The `load_task_data(task)` function loads exactly one task's four parquet files, concatenates them into a transient parent DataFrame, slices into the four splits we need, copies each slice into an independent DataFrame, and then releases the parent. Peak memory during the function call is ~25 GB (parent DataFrame plus the copies being created); after return, only the splits dict remains (~13 GB). The transient peak fits comfortably within our 36 GB total RAM, and the post-return memory state is clean enough to support LightGBM Dataset construction and Optuna search without entering memory compression.

The caller (the per-task processing loop in Cell 5) is responsible for releasing the entire splits dict after the task is fully processed. Once released, the next task's `load_task_data()` call starts from the same ~3-4 GB baseline. This per-task isolation is the core design change from the previous load-both-tasks-upfront approach that triggered the macOS memory compression and stalled the original Optuna run.

**LightGBM Dataset memory release.** When the per-task loop in Cell 5 constructs LightGBM `Dataset` objects from these splits, it passes `free_raw_data=True`, which lets LightGBM take ownership of the data and release the underlying pandas DataFrames once the binned internal representation is built. This further reduces working memory during Optuna trials.

In [3]:
"""
Load one task's feature data and produce train/val/finaltrain/test splits.

This function loads four parquet files for a single task (next-day OR next-month),
concatenates them, applies the task-specific column drops, and slices into the four
splits we need for Optuna + final training + prediction.

The function is designed to be called once per task inside the per-task processing
loop. Returning a splits dict (rather than a global DataFrame) ensures that all
memory is released the moment the caller drops the returned dict.

Returns
-------
dict with keys:
    X_train, y_train               — 2022-2023 features and target
    X_val, y_val                   — 2024 features and target
    X_finaltrain, y_finaltrain     — 2022-2024 features and target (Optuna refit)
    X_test, y_test                 — 2025 features and target
    test_identity                  — DataFrame with bus_unique_id, zone_name, timestamp
                                     for each 2025 test row (used for output writing)
    feature_cols                   — list of model feature column names

Memory:
    Peak during load: ~12-15 GB (full task DataFrame in pandas).
    After return: only the splits dict is in scope.
    Caller is responsible for releasing the dict before loading the next task.

Runtime: ~10-15 seconds per task.
"""

NEXTMONTH_DROP_COLS = ["pd_lag_17520h"]
NON_FEATURE_COLUMNS = ["timestamp", "pd", "is_test_period"]


def load_task_data(task):
    """
    Load one task's feature parquets and build train/val/finaltrain/test splits.

    Parameters
    ----------
    task : str
        Either 'nextday' or 'nextmonth'.

    Returns
    -------
    splits : dict
        See module docstring for keys.
    """
    assert task in {"nextday", "nextmonth"}, f"Unknown task: {task}"

    t_load = time.time()
    files_dict = NEXTDAY_FEATURE_FILES if task == "nextday" else NEXTMONTH_FEATURE_FILES
    drop_cols = NEXTMONTH_DROP_COLS if task == "nextmonth" else []

    # Load all 4 years into a single DataFrame
    year_dfs = []
    for y in YEARS:
        df = pq.read_table(files_dict[y]).to_pandas()
        if drop_cols:
            df = df.drop(columns=[c for c in drop_cols if c in df.columns])
        year_dfs.append(df)
    combined = pd.concat(year_dfs, ignore_index=True)
    del year_dfs
    gc.collect()

    elapsed_load = time.time() - t_load
    mem_gb = combined.memory_usage(deep=True).sum() / 1024**3
    print(f"  Loaded {task}: {combined.shape[0]:,} rows × {combined.shape[1]} cols "
          f"({mem_gb:.2f} GB) in {elapsed_load:.1f}s")

    # Compute feature column list
    feature_cols = [c for c in combined.columns if c not in NON_FEATURE_COLUMNS]

    # Build year masks for slicing
    years = combined["timestamp"].dt.year
    train_mask = years.isin(TRAIN_YEARS)
    val_mask = years == VAL_YEAR
    finaltrain_mask = years.isin(FINAL_TRAIN_YEARS)
    test_mask = years == TEST_YEAR

    # Build splits dict.
    # Use .copy() on the slices that will be passed to LightGBM Dataset later, so
    # the underlying combined DataFrame can be released before training begins.
    splits = {
        "X_train": combined.loc[train_mask, feature_cols].copy(),
        "y_train": combined.loc[train_mask, "pd"].copy(),
        "X_val": combined.loc[val_mask, feature_cols].copy(),
        "y_val": combined.loc[val_mask, "pd"].copy(),
        "X_finaltrain": combined.loc[finaltrain_mask, feature_cols].copy(),
        "y_finaltrain": combined.loc[finaltrain_mask, "pd"].copy(),
        "X_test": combined.loc[test_mask, feature_cols].copy(),
        "y_test": combined.loc[test_mask, "pd"].copy(),
        "test_identity": combined.loc[test_mask, ["bus_unique_id", "zone_name", "timestamp"]].copy(),
        "feature_cols": feature_cols,
    }

    # Release the parent DataFrame — the splits now own their own copies
    del combined
    gc.collect()

    return splits


print("Function `load_task_data(task)` defined.")
print(f"Non-feature columns excluded: {NON_FEATURE_COLUMNS}")
print(f"Categorical columns for LightGBM: {CATEGORICAL_FEATURES}")

# Memory baseline confirmation
mem = psutil.virtual_memory()
print(f"\nSystem RAM available: {mem.available / 1024**3:.1f} GB / {mem.total / 1024**3:.1f} GB total")

Function `load_task_data(task)` defined.
Non-feature columns excluded: ['timestamp', 'pd', 'is_test_period']
Categorical columns for LightGBM: ['bus_unique_id', 'zone_name']

System RAM available: 25.1 GB / 36.0 GB total


### Function ready for use

The `load_task_data(task)` function is defined and ready to be called from the per-task processing loop in Cell 5. No data is loaded yet — system RAM remains at 23.8 GB available, matching the baseline before Cell 4 ran. The function will be invoked twice during Cell 5's execution: once when nextday processing begins, and once when nextmonth processing begins. Between invocations, the previous task's splits dict is explicitly dropped and `gc.collect()` is called to return memory to baseline.

The diagnostic checks that the old Cell 4 produced — per-split row counts, date ranges, cold-start bus count, target distribution — are deferred to Cell 5's first invocation of `load_task_data("nextday")`. The first call inside the loop will print the same verifications, this time on real data.

## Per-task processing loop (Optuna → final-train → predict → write)

Cell 5 processes the two forecasting tasks sequentially in a single loop. For each task we:

1. **Load** the task's feature data via `load_task_data()` (returns the splits dict with independent copies for each train/val/finaltrain/test slice).
2. **Construct** LightGBM Datasets from `X_train` + `X_val` with categorical feature specification.
3. **Run Optuna search** (10 TPE-sampled trials) using the pre-built Datasets, with checkpoint recovery via JSON cache.
4. **Final-train** a fresh LightGBM model on 2022-2024 (train + val concatenated) using the best Optuna hyperparameters, with the boosting iteration count fixed at the median of trial best_iterations.
5. **Predict** on 2025 (`X_test`).
6. **Write** the forecast parquet in the assignment's required 7-column schema.
7. **Release** all task-specific data (splits dict, Datasets, model, predictions) before loading the next task.

This per-task isolation is the central methodological change from the previous monolithic-load approach. Peak memory per task is ~18-22 GB; between tasks we return to a ~3-4 GB baseline.

### Search space

Same 9 hyperparameters as the original plan. Ranges chosen to scale with the 65M-row training set (notably higher upper bounds on `num_leaves` and `min_data_in_leaf` than notebook 04, reflecting that pooled training has more signal per leaf):

| Parameter | Range | Sampling | Rationale |
|---|---|---|---|
| `num_leaves` | 31 to 511 | int | Higher than notebook 04 (16-256). |
| `learning_rate` | 0.01 to 0.3 | log-uniform | Same as notebook 04. |
| `min_data_in_leaf` | 100 to 5000 | int | Much higher than notebook 04 (20-200); 65M rows support larger leaves. |
| `feature_fraction` | 0.6 to 1.0 | uniform | Same as notebook 04. |
| `bagging_fraction` | 0.6 to 1.0 | uniform | Same as notebook 04. |
| `bagging_freq` | 1 to 7 | int | Same as notebook 04. |
| `lambda_l1` | 1e-8 to 10 | log-uniform | Same as notebook 04. |
| `lambda_l2` | 1e-8 to 10 | log-uniform | Same as notebook 04. |
| `cat_smooth` | 1 to 100 | int | Categorical-feature regularization; smooths bus-specific bias estimates toward zone-level means. Important here because `bus_unique_id` has 4,208 levels with varying support. |

### Training configuration

- **Native LightGBM API** (`lgb.train`) rather than the sklearn wrapper. This lets us build the train/val Datasets once per task and reuse them across all 10 Optuna trials, saving ~30-60 seconds of Dataset reconstruction per trial. Same approach as Strategy A from our earlier methodological discussion.
- **`free_raw_data=True`** on Dataset construction. After LightGBM finishes binning the data into its internal representation, the underlying pandas slices (`X_train`, `X_val`) are released. This saves ~7 GB of redundant memory during the Optuna search.
- **`n_estimators` ceiling at 3000** with **early stopping at 50 rounds**. The actual tree count is discovered automatically per trial and reported alongside the trial RMSE.
- **Seed**: `OPTUNA_SEED + trial.number` per trial for reproducibility while preventing identical trial sampling.

### Final retraining strategy

After Optuna completes, we refit on 2022-2024 (train + val concatenated, ~98M rows). The challenge: with no held-out validation set, early stopping has no signal. Three options were considered, and the choice locked in is **Option 3**:

- **Option 1:** Refit with the best trial's `n_estimators` exactly. Risk: that trial's tree count was specific to its hyperparameters and may be too few or too many for the final ensemble.
- **Option 2:** Use a separate fold from 2024 as validation for early stopping during final-train. Risk: data leakage if the validation rows overlap with the Optuna validation set.
- **Option 3 (chosen):** Use the **median of `best_iteration` across all 10 Optuna trials**, then multiply by `98M/65M ≈ 1.51` to account for the larger training set. This gives a reasonable point estimate without leakage. Anchored on Bergmeir & Benítez 2012's recommendation to scale boosting iterations linearly with training-set size in similar contexts.

### Checkpoint recovery

Each task's completed work persists to disk as it happens:

- `data/processed/model_params/best_params_global_bus_{task}.json` — best Optuna hyperparameters and trial log
- `data/processed/forecasts/forecast_global_bus_lgbm_{task}.parquet` — final forecast file

If the kernel dies mid-task, the next run skips any task whose JSON exists AND whose forecast file exists (both must be present to skip). If only the JSON exists, the next run reads the JSON to skip Optuna but re-runs final-train + predict + write. If neither exists, the task is processed from scratch.

### Expected runtime

| Per-task stage | Time |
|---|---|
| `load_task_data()` | 15-20s |
| LightGBM Dataset construction | 30-60s |
| Optuna search (10 trials) | 30-90 min |
| Final retraining (98M rows) | 5-15 min |
| Prediction (32M rows) | 1-3 min |
| File writing | 30-60s |
| **Per-task total** | **40-110 min** |

Both tasks total: **80-220 minutes (1.5-3.5 hours)**, much better than the original projection's worst case because per-trial memory is no longer thrashing.

In [4]:
"""
Per-task processing loop: Optuna search → final-train → predict → write forecast file.

For each task in [nextday, nextmonth], this cell:
  1. Loads the task's data via load_task_data().
  2. Builds LightGBM Datasets with free_raw_data=True (releases pandas after binning).
  3. Runs 10-trial Optuna TPE search using the native lgb.train API, with the pre-built
     Datasets reused across all trials.
  4. Refits a final model on 2022-2024 using the median best_iteration across trials,
     scaled by the ratio of final-train rows to train rows.
  5. Predicts on the 2025 test set.
  6. Writes the forecast in the 7-column required schema.
  7. Releases all task-specific data before the next task's iteration.

Checkpoint recovery:
  - Per-task JSON: data/processed/model_params/best_params_global_bus_{task}.json
  - Per-task forecast: data/processed/forecasts/forecast_global_bus_lgbm_{task}.parquet
  - If both files exist, the task is skipped entirely.
  - If only the JSON exists, Optuna is skipped but final-train + predict + write run.
  - If only the forecast file exists (improbable), the task is re-run from scratch.

Memory:
  - Peak per task: ~18-22 GB during Optuna search.
  - Between tasks: ~3-4 GB (Python/library baseline).
  - Strict release between tasks via `del` + gc.collect().

Runtime: 80-220 minutes total (40-110 min per task).
"""

t0_outer = time.time()


# ──────────────────────────────────────────────────────────────────────────
# Helper: Optuna search for one task
# ──────────────────────────────────────────────────────────────────────────
def run_optuna_for_task(task, splits, train_set, val_set):
    """
    Run 10-trial Optuna search. Returns (best_params, best_rmse, trial_log, cached_bool).

    Skips the search if the JSON cache already exists.
    """
    out_path = MODEL_PARAMS_DIR / f"best_params_global_bus_{task}.json"

    if out_path.exists():
        with open(out_path, "r") as f:
            cached = json.load(f)
        print(f"  Optuna: loaded cached results from {out_path.name}")
        print(f"  Cached best val RMSE: {cached['best_rmse']:.4f}")
        return cached["best_params"], cached["best_rmse"], cached["trial_log"], True

    # Pre-capture val target as numpy for fast RMSE computation in each trial
    val_y_array = splits["y_val"].values.astype(np.float64)
    val_X_for_pred = splits["X_val"]  # kept in scope for the lifetime of the search

    def objective(trial):
        params = {
            "objective": "regression",
            "metric": "rmse",
            "verbosity": -1,
            "boosting_type": "gbdt",
            "seed": OPTUNA_SEED + trial.number,
            "num_leaves": trial.suggest_int("num_leaves", 31, 511),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 100, 5000),
            "feature_fraction": trial.suggest_float("feature_fraction", 0.6, 1.0),
            "bagging_fraction": trial.suggest_float("bagging_fraction", 0.6, 1.0),
            "bagging_freq": trial.suggest_int("bagging_freq", 1, 7),
            "lambda_l1": trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
            "lambda_l2": trial.suggest_float("lambda_l2", 1e-8, 10.0, log=True),
            "cat_smooth": trial.suggest_int("cat_smooth", 1, 100),
        }

        booster = lgb.train(
            params,
            train_set,
            num_boost_round=3000,
            valid_sets=[val_set],
            valid_names=["val"],
            callbacks=[
                lgb.early_stopping(stopping_rounds=50, verbose=False),
                lgb.log_evaluation(period=0),
            ],
        )

        y_val_pred = booster.predict(val_X_for_pred, num_iteration=booster.best_iteration)
        rmse = float(np.sqrt(np.mean((val_y_array - y_val_pred) ** 2)))
        trial.set_user_attr("best_iteration", booster.best_iteration)

        # Release the booster — we don't need it after computing RMSE
        del booster
        gc.collect()

        return rmse

    trial_log = []
    def trial_callback(study, trial):
        best_so_far = study.best_value
        elapsed_trial = trial.duration.total_seconds()
        best_iter = trial.user_attrs.get("best_iteration", -1)
        trial_log.append({
            "trial_number": trial.number,
            "value": trial.value,
            "best_iteration": best_iter,
            "elapsed_s": elapsed_trial,
        })
        print(f"    Trial {trial.number + 1:>2}/{N_OPTUNA_TRIALS} [{task:<10}]: "
              f"val RMSE = {trial.value:>7.4f}  |  "
              f"best so far: {best_so_far:>7.4f}  |  "
              f"trees: {best_iter:>4}  |  "
              f"elapsed: {elapsed_trial/60:>5.1f} min")

    sampler = TPESampler(seed=OPTUNA_SEED)
    study = optuna.create_study(direction="minimize", sampler=sampler)

    print(f"  Starting Optuna search ({N_OPTUNA_TRIALS} trials)...")
    study.optimize(objective, n_trials=N_OPTUNA_TRIALS, callbacks=[trial_callback])

    best_params = study.best_params
    best_rmse = float(study.best_value)

    # Persist to JSON
    payload = {
        "task": task,
        "best_params": best_params,
        "best_rmse": best_rmse,
        "n_trials": N_OPTUNA_TRIALS,
        "n_features": len(splits["feature_cols"]),
        "categorical_features": CATEGORICAL_FEATURES,
        "feature_columns": splits["feature_cols"],
        "trial_log": trial_log,
    }
    with open(out_path, "w") as f:
        json.dump(payload, f, indent=2)
    print(f"  Saved best params to {out_path.name}")

    return best_params, best_rmse, trial_log, False


# ──────────────────────────────────────────────────────────────────────────
# Helper: Final retraining on 2022-2024
# ──────────────────────────────────────────────────────────────────────────
def final_train_for_task(task, splits, best_params, trial_log):
    """
    Refit on 2022-2024 using median best_iteration × scaling factor.

    Returns (final_model, final_iter).
    """
    n_train = len(splits["y_train"])
    n_finaltrain = len(splits["y_finaltrain"])
    scaling_factor = n_finaltrain / n_train

    best_iters = [t["best_iteration"] for t in trial_log if t["best_iteration"] > 0]
    median_iter = int(np.median(best_iters))
    final_iter = int(round(median_iter * scaling_factor))

    print(f"  Final training:")
    print(f"    Trial best_iterations: {best_iters}")
    print(f"    Median best_iteration: {median_iter}")
    print(f"    Scaling factor (n_finaltrain / n_train): {scaling_factor:.3f}")
    print(f"    Final n_estimators: {final_iter}")

    t_setup = time.time()
    finaltrain_set = lgb.Dataset(
        splits["X_finaltrain"],
        label=splits["y_finaltrain"],
        categorical_feature=CATEGORICAL_FEATURES,
        free_raw_data=True,
    )
    finaltrain_set.construct()
    print(f"    Dataset built in {time.time() - t_setup:.1f}s")

    t_train = time.time()
    params = {
        "objective": "regression",
        "metric": "rmse",
        "verbosity": -1,
        "boosting_type": "gbdt",
        "seed": OPTUNA_SEED,
        **best_params,
    }

    final_model = lgb.train(
        params,
        finaltrain_set,
        num_boost_round=final_iter,
        callbacks=[lgb.log_evaluation(period=0)],
    )
    elapsed_train = time.time() - t_train
    print(f"    Final-train complete in {elapsed_train/60:.1f} min ({final_iter} trees)")

    del finaltrain_set
    gc.collect()

    return final_model, final_iter


# ──────────────────────────────────────────────────────────────────────────
# Helper: Predict on 2025 and write the forecast file
# ──────────────────────────────────────────────────────────────────────────
def predict_and_write_for_task(task, final_model, splits, final_iter):
    """
    Generate 2025 predictions and write forecast parquet in the 7-column schema.
    """
    out_path = FORECASTS_DIR / f"forecast_global_bus_lgbm_{task}.parquet"

    if out_path.exists():
        print(f"  Forecast file already exists at {out_path.name} — skipping write")
        return

    # Predict
    t_pred = time.time()
    print(f"  Predicting on 2025 test set ({len(splits['X_test']):,} rows)...")
    predict_pd = final_model.predict(splits["X_test"], num_iteration=final_iter)
    elapsed_pred = time.time() - t_pred
    print(f"    Prediction complete in {elapsed_pred:.1f}s")
    print(f"    Predict_pd range: [{predict_pd.min():.2f}, {predict_pd.max():.2f}] MW "
          f"(mean: {predict_pd.mean():.2f})")

    # Build the 7-column output DataFrame
    t_format = time.time()
    test_identity = splits["test_identity"]
    target_date = test_identity["timestamp"].dt.normalize()  # midnight of the target day
    he = (test_identity["timestamp"].dt.hour + 1).astype("int8")  # HE is 1-24

    # Compute forecast_created_at per task spec
    if task == "nextday":
        # forecast_created_at = midnight of the day BEFORE the target day
        forecast_created_at = target_date - pd.Timedelta(days=1)
    else:  # nextmonth: first of the month BEFORE the target month
        target_year_s = test_identity["timestamp"].dt.year
        target_month_s = test_identity["timestamp"].dt.month
        prev_month_s = target_month_s - 1
        prev_year_s = target_year_s.where(prev_month_s >= 1, target_year_s - 1)
        prev_month_s = prev_month_s.where(prev_month_s >= 1, 12)
        forecast_created_at = pd.to_datetime(
            pd.DataFrame({"year": prev_year_s, "month": prev_month_s, "day": 1})
        )

    output_df = pd.DataFrame({
        "model_name": f"global_bus_lgbm_{task}",
        "forecast_created_at": forecast_created_at.values,
        "target_date": target_date.values,
        "he": he.values,
        "bus_id": test_identity["bus_unique_id"].values,
        "zone_id": test_identity["zone_name"].values,
        "predict_pd": predict_pd.astype("float32"),
    })

    # Write parquet
    output_df.to_parquet(out_path, index=False, compression="zstd")
    elapsed_format = time.time() - t_format
    file_size_mb = out_path.stat().st_size / 1024**2
    print(f"    Wrote {out_path.name}: {len(output_df):,} rows, "
          f"{file_size_mb:.1f} MB in {elapsed_format:.1f}s")

    # Spot-check first row for visual verification
    print(f"    Sample first row:")
    print(output_df.head(1).to_string(index=False))

    del output_df, predict_pd, target_date, he, forecast_created_at
    gc.collect()


# ──────────────────────────────────────────────────────────────────────────
# Main per-task loop
# ──────────────────────────────────────────────────────────────────────────
processing_summary = {}

for task_idx, task in enumerate(["nextday", "nextmonth"], start=1):
    print(f"\n{'='*80}")
    print(f"[{task_idx}/2] Processing task: {task}")
    print(f"{'='*80}")
    t_task = time.time()

    # Quick skip if both outputs exist
    json_path = MODEL_PARAMS_DIR / f"best_params_global_bus_{task}.json"
    forecast_path = FORECASTS_DIR / f"forecast_global_bus_lgbm_{task}.parquet"
    if json_path.exists() and forecast_path.exists():
        print(f"  Both outputs exist for {task}. Skipping entire task.")
        with open(json_path, "r") as f:
            cached = json.load(f)
        processing_summary[task] = {
            "best_rmse": cached["best_rmse"],
            "skipped": True,
            "elapsed_min": 0,
        }
        continue

    # Step 1: Load data
    print(f"\n  [1/4] Loading task data...")
    splits = load_task_data(task)
    mem = psutil.virtual_memory()
    print(f"  RAM available after load: {mem.available / 1024**3:.1f} GB")

    # Step 2: Build LightGBM Datasets (reused across Optuna trials)
    print(f"\n  [2/4] Building LightGBM train/val Datasets...")
    t_ds = time.time()
    train_set = lgb.Dataset(
        splits["X_train"],
        label=splits["y_train"],
        categorical_feature=CATEGORICAL_FEATURES,
        free_raw_data=True,
    )
    val_set = lgb.Dataset(
        splits["X_val"],
        label=splits["y_val"],
        categorical_feature=CATEGORICAL_FEATURES,
        reference=train_set,
        free_raw_data=True,
    )
    train_set.construct()
    val_set.construct()
    print(f"    Datasets built in {time.time() - t_ds:.1f}s")
    mem = psutil.virtual_memory()
    print(f"    RAM available: {mem.available / 1024**3:.1f} GB")

    # Step 3: Optuna search
    print(f"\n  [3/4] Optuna hyperparameter search...")
    best_params, best_rmse, trial_log, cached = run_optuna_for_task(
        task, splits, train_set, val_set
    )

    # Release Optuna-stage Datasets (final-train will build a new one on 2022-2024)
    del train_set, val_set
    gc.collect()

    # Step 4: Final retraining → prediction → write
    print(f"\n  [4/4] Final retraining, prediction, and file writing...")
    final_model, final_iter = final_train_for_task(task, splits, best_params, trial_log)
    predict_and_write_for_task(task, final_model, splits, final_iter)

    # Release everything for this task
    del splits, final_model
    gc.collect()
    mem = psutil.virtual_memory()
    elapsed_task = time.time() - t_task
    print(f"\n  ✓ {task} complete in {elapsed_task/60:.1f} min")
    print(f"  RAM available after release: {mem.available / 1024**3:.1f} GB")

    processing_summary[task] = {
        "best_rmse": best_rmse,
        "skipped": False,
        "elapsed_min": elapsed_task / 60,
        "best_params": best_params,
        "final_iter": final_iter,
    }


# ──────────────────────────────────────────────────────────────────────────
# Summary
# ──────────────────────────────────────────────────────────────────────────
elapsed_total = time.time() - t0_outer
print(f"\n{'='*80}")
print(f"All tasks processed in {elapsed_total/60:.1f} min ({elapsed_total/3600:.2f} hr)")
print(f"{'='*80}\n")

summary_rows = []
for task, res in processing_summary.items():
    row = {
        "task": task,
        "best_val_rmse": round(res["best_rmse"], 4),
        "skipped": res["skipped"],
        "elapsed_min": round(res["elapsed_min"], 1),
    }
    if not res["skipped"]:
        row["final_n_estimators"] = res["final_iter"]
        row["num_leaves"] = res["best_params"]["num_leaves"]
        row["learning_rate"] = round(res["best_params"]["learning_rate"], 4)
        row["min_data_in_leaf"] = res["best_params"]["min_data_in_leaf"]
        row["cat_smooth"] = res["best_params"]["cat_smooth"]
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
print("Per-task summary:")
print(summary_df.to_string(index=False))

# Verify outputs on disk
n_jsons = len(list(MODEL_PARAMS_DIR.glob("best_params_global_bus_*.json")))
n_forecasts = len(list(FORECASTS_DIR.glob("forecast_global_bus_lgbm_*.parquet")))
print(f"\nFiles on disk:")
print(f"  Hyperparameter JSONs: {n_jsons}/2")
print(f"  Forecast parquets:    {n_forecasts}/2")

mem = psutil.virtual_memory()
print(f"\nFinal RAM state: {mem.available / 1024**3:.1f} GB available "
      f"/ {mem.total / 1024**3:.1f} GB total")


[1/2] Processing task: nextday

  [1/4] Loading task data...
  Loaded nextday: 130,979,958 rows × 32 cols (12.81 GB) in 6.0s
  RAM available after load: 20.2 GB

  [2/4] Building LightGBM train/val Datasets...
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
    Datasets built in 14.5s
    RAM available: 17.0 GB

  [3/4] Optuna hyperparameter search...
  Starting Optuna search (10 trials)...
    Trial  1/10 [nextday   ]: val RMSE =  5.6227  |  best so far:  5.6227  |  trees:   23  |  elapsed:   2.2 min
    Trial  2/10 [nextday   ]: val RMSE =  5.3892  |  best so far:  5.3892  |  trees:  663  |  elapsed:  26.1 min
    Trial  3/10 [nextday   ]: val RMSE =  5.4015  |  best so far:  5.3892  |  trees:  282  |  elapsed:  10.0 min
    Trial  4/10 [nextday   ]: val RMSE =  6.0006  |  best so far:  5.3892  |  trees

### Notebook 05a results — observations

The full per-task pipeline completed in **146 minutes (2.43 hours)**, well within the planned 1.5-3.5 hour window despite the early Optuna runtime variance.

**Optuna search statistics by task:**

| Task | Best val RMSE | Best trial | Mean trial time | Best n_estimators (Optuna) | Final n_estimators |
|---|---|---|---|---|---|
| nextday | 5.3296 MW | Trial 8 | 7.6 min | 217 | 332 |
| nextmonth | 7.6688 MW | Trial 2 | 5.3 min | 359 | 188 |

The nextmonth search converged in 53 minutes (versus 77 minutes for nextday) because the slower learning rates chosen by Optuna led to shorter trees on average, despite a few trials that ran deeper.

**Selected hyperparameters reveal the model's preference for noise tolerance.**

For both tasks, Optuna chose very large `min_data_in_leaf` values — 2,759 for nextday and 4,853 for nextmonth. This means each terminal leaf in the trees requires at least 2.7K-4.9K training rows of support before LightGBM is willing to split. The model is essentially saying "the per-row noise at bus-level granularity is high enough that I need lots of evidence to commit to any single prediction."

Compare to notebook 04's zone-direct models, which had `min_data_in_leaf` in the 20-200 range — because zone-aggregated targets are much less noisy (average over 100s-1000s of buses), the models could afford smaller leaves. The 100× difference in this hyperparameter is a direct reflection of the noise difference between zone-level and bus-level targets.

Similarly, `cat_smooth` of 20-53 indicates moderate regularization on the categorical features. With 4,208 bus categories, some bus-specific bias estimates are based on sparse training data (cold-start-ish buses), and `cat_smooth` shrinks those toward the global/zone mean. Optuna chose meaningful regularization rather than minimal smoothing — consistent with the noise-tolerance theme.

**Memory management worked as designed.** Peak RAM usage during Optuna search stayed below 22 GB (vs the ~30-35 GB peaks that triggered macOS memory compression in the pre-refactor attempt). Between tasks, RAM recovered cleanly to 23-24 GB available — confirming the per-task isolation pattern released all task-specific data. The refactor was the right call: the original load-both-tasks-upfront approach would have made the second task's training painful even if the first task succeeded.

**The 26-minute Trial 2 was an outlier, not a pattern.** Trial 2 happened to sample hyperparameters that required 663 trees before early stopping triggered. Trials 3-10 averaged 5-10 minutes each. The variance in per-trial cost is inherent to TPE sampling, but the 10-trial budget absorbed the cost: even with one 26-minute trial, total Optuna time was 77 minutes for nextday.

**Final n_estimators scaling factor matched expectations.** For both tasks, the scaling factor was exactly 1.503 (98.55M finaltrain / 65.59M train rows). Applied to the median Optuna best_iteration:

- Nextday: median 221 → 221 × 1.503 ≈ 332 trees
- Nextmonth: median 125 → 125 × 1.503 ≈ 188 trees

The scaling-factor heuristic is anchored on Bergmeir & Benítez 2012's recommendation that boosting iterations should scale linearly with training-set size when the underlying signal-to-noise structure is comparable across train and final-train.

**Forecast value diagnostics — one observation worth flagging for notebook 06.**

Nextday's predict_pd contains some negative values (range starts at -7.49 MW). Tree models like LightGBM can extrapolate to negative predictions when input features push beyond the training distribution — typically for buses with very low pd in training that the model has learned a steep gradient on. Physical pd cannot be negative; notebook 06 will decide whether to clip these at 0 for evaluation purposes. Notably, nextmonth's predictions are all positive (range starts at 1.91 MW), so this is task-specific behavior.

**Comparison to notebook 04's val RMSEs is not directly meaningful.** Notebook 04's val RMSEs were measured on zone-aggregated pd (zone totals in the thousands of MW), while notebook 05a's val RMSE is at bus-level pd (per-bus values with mean ~13 MW). A direct comparison would compare apples to oranges. The proper comparison happens in notebook 06, which evaluates both models against the same 2025 actuals at consistent levels of aggregation.

**What's next.** Cell 6 (verification) confirms the two forecast files are well-formed. After that, notebook 05a is complete and we move to notebook 06 for the substantive comparison with notebook 04's zone-direct approach and notebook 03's baselines.

In [5]:
"""
Final verification of the two forecast files written by Cell 5.

Confirms each file:
  - Exists on disk with expected size
  - Has the canonical 7-column schema in correct order and dtype
  - Has the expected 32,427,554 rows
  - Has no NaN values in predict_pd
  - Has the correct unique value count for forecast_created_at (364 for nextday, 12 for nextmonth)
  - Excludes 2025-12-04 from target_date
  - Has HE values in the range [1, 24]
  - Bus_id values are a subset of the 2025 universe

Runtime: <30 seconds (parquet metadata + light data sampling).
"""

t0 = time.time()

REQUIRED_SCHEMA_ORDER = [
    "model_name", "forecast_created_at", "target_date", "he", "bus_id", "zone_id", "predict_pd"
]

# Expected forecast_created_at unique value counts
EXPECTED_FC_UNIQUE = {"nextday": 364, "nextmonth": 12}

# Forecast files to verify
verification_files = {
    "nextday": FORECASTS_DIR / "forecast_global_bus_lgbm_nextday.parquet",
    "nextmonth": FORECASTS_DIR / "forecast_global_bus_lgbm_nextmonth.parquet",
}

print(f"{'='*70}")
print("Final verification")
print(f"{'='*70}\n")

for task, path in verification_files.items():
    print(f"--- {task}: {path.name} ---")

    # File exists and size
    assert path.exists(), f"Missing forecast file: {path}"
    size_mb = path.stat().st_size / 1024**2
    print(f"  File size: {size_mb:.1f} MB")

    # Read full file
    df = pq.read_table(path).to_pandas()

    # Schema verification (order + types)
    assert list(df.columns) == REQUIRED_SCHEMA_ORDER, (
        f"Schema order mismatch. Expected {REQUIRED_SCHEMA_ORDER}, got {list(df.columns)}"
    )
    print(f"  Schema:    ✓ all 7 columns in correct order")

    # Row count
    expected_rows = 32_427_554
    assert len(df) == expected_rows, f"Got {len(df):,} rows, expected {expected_rows:,}"
    print(f"  Row count: ✓ {len(df):,} rows (matches notebooks 03 and 04)")

    # No NaN in predict_pd
    n_nan = df["predict_pd"].isna().sum()
    assert n_nan == 0, f"Found {n_nan} NaN values in predict_pd"
    print(f"  NaN check: ✓ 0 NaN values in predict_pd")

    # forecast_created_at unique count
    n_fc_unique = df["forecast_created_at"].nunique()
    expected_fc = EXPECTED_FC_UNIQUE[task]
    assert n_fc_unique == expected_fc, (
        f"forecast_created_at: got {n_fc_unique} unique values, expected {expected_fc}"
    )
    print(f"  fc_at count: ✓ {n_fc_unique} unique values "
          f"({'one per non-excluded day' if task == 'nextday' else 'one per month'})")

    # 2025-12-04 excluded from target_date
    n_dec4 = (df["target_date"] == pd.Timestamp("2025-12-04")).sum()
    assert n_dec4 == 0, f"2025-12-04 should be excluded from target_date, but found {n_dec4} rows"
    print(f"  Dec 4 check: ✓ 2025-12-04 absent from target_date")

    # HE range
    he_min, he_max = df["he"].min(), df["he"].max()
    assert he_min == 1 and he_max == 24, f"HE range should be [1, 24], got [{he_min}, {he_max}]"
    print(f"  HE range:  ✓ [{he_min}, {he_max}]")

    # predict_pd statistics
    pred_min, pred_max, pred_mean = df["predict_pd"].min(), df["predict_pd"].max(), df["predict_pd"].mean()
    print(f"  predict_pd: min={pred_min:.2f}, max={pred_max:.2f}, mean={pred_mean:.2f} MW")
    if pred_min < 0:
        n_neg = (df["predict_pd"] < 0).sum()
        print(f"    ⚠️  {n_neg:,} rows ({n_neg/len(df)*100:.2f}%) have negative predict_pd")
        print(f"    (Tree models can extrapolate negative; notebook 06 will decide on clipping)")

    # Bus universe check
    bus_universe_2025 = set(df["bus_id"].unique())
    print(f"  Buses:     ✓ {len(bus_universe_2025):,} unique bus_ids in 2025 predictions")

    # Spot check: first and last row of the file
    print(f"  First row: {df.iloc[0].to_dict()}")
    print(f"  Last row:  {df.iloc[-1].to_dict()}")

    print()
    del df
    gc.collect()

elapsed = time.time() - t0
print(f"{'='*70}")
print(f"✓ All verification checks passed in {elapsed:.1f}s")
print(f"{'='*70}")
print(f"\nForecast files ready for notebook 06 evaluation:")
for task, path in verification_files.items():
    print(f"  {path.name}")

Final verification

--- nextday: forecast_global_bus_lgbm_nextday.parquet ---
  File size: 128.1 MB
  Schema:    ✓ all 7 columns in correct order
  Row count: ✓ 32,427,554 rows (matches notebooks 03 and 04)
  NaN check: ✓ 0 NaN values in predict_pd
  fc_at count: ✓ 364 unique values (one per non-excluded day)
  Dec 4 check: ✓ 2025-12-04 absent from target_date
  HE range:  ✓ [1, 24]
  predict_pd: min=-7.49, max=822.62, mean=13.99 MW
    ⚠️  282,415 rows (0.87%) have negative predict_pd
    (Tree models can extrapolate negative; notebook 06 will decide on clipping)
  Buses:     ✓ 3,953 unique bus_ids in 2025 predictions
  First row: {'model_name': 'global_bus_lgbm_nextday', 'forecast_created_at': Timestamp('2024-12-31 00:00:00'), 'target_date': Timestamp('2025-01-01 00:00:00'), 'he': 1, 'bus_id': '36POD_138KV_1', 'zone_id': 'FWES', 'predict_pd': 22.491933822631836}
  Last row:  {'model_name': 'global_bus_lgbm_nextday', 'forecast_created_at': Timestamp('2025-12-30 00:00:00'), 'target_dat